<a href="https://colab.research.google.com/github/pinkprincess536/eeg/blob/main/eeg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mne numpy pandas matplotlib scikit-learn torch

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import os, re, subprocess, time
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 1: Config

All tunable parameters. Seed=42 for reproducible split.

In [ ]:
BASE_DIR = "/content/drive/MyDrive/EEG_PROJECT"
os.makedirs(BASE_DIR, exist_ok=True)

# 11 patients in serial order, shuffled with seed=42
ALL_PATIENTS = [f"chb{i:02d}" for i in range(1, 12)]
np.random.seed(42)
shuffled = ALL_PATIENTS.copy()
np.random.shuffle(shuffled)
TRAIN_PATIENTS = sorted(shuffled[:9])
TEST_PATIENTS  = sorted(shuffled[9:])

print(f"All {len(ALL_PATIENTS)} patients: {ALL_PATIENTS}")
print(f"Train ({len(TRAIN_PATIENTS)}): {TRAIN_PATIENTS}")
print(f"Test  ({len(TEST_PATIENTS)}): {TEST_PATIENTS}")

# Pipeline params
WINDOW_SIZE_SEC = 7.0
OVERLAP         = 0.3
LOWCUT          = 0.5
HIGHCUT         = 40.0
NOTCH_FREQ      = 60.0

# Training params
BATCH_SIZE           = 8
LEARNING_RATE        = 0.001
NUM_EPOCHS           = 10
RANDOM_SEED          = 42
NON_SEIZURE_SAMPLES  = 500

print(f"Window: {WINDOW_SIZE_SEC}s | Overlap: {OVERLAP} | Filter: {LOWCUT}-{HIGHCUT} Hz")
print(f"Batch: {BATCH_SIZE} | LR: {LEARNING_RATE} | Epochs: {NUM_EPOCHS} | Non-seizure: {NON_SEIZURE_SAMPLES}")

## Cell 2: Download Data

6 parallel downloads. Skips files already on Drive.

In [ ]:
start_time = time.time()
total_bytes = 0

for i in range(1, 12):
    pid = f"chb{i:02d}"
    pdir = os.path.join(BASE_DIR, pid)
    os.makedirs(pdir, exist_ok=True)

    # Download summary
    summary_url = f"https://physionet.org/files/chbmit/1.0.0/{pid}/{pid}-summary.txt"
    summary_path = os.path.join(BASE_DIR, f"{pid}-summary.txt")
    if not os.path.exists(summary_path):
        subprocess.run(["wget", "-q", "-O", summary_path, summary_url])
    total_bytes += os.path.getsize(summary_path)

    # Download RECORDS
    records_url = f"https://physionet.org/files/chbmit/1.0.0/{pid}/RECORDS"
    records_path = os.path.join(pdir, "RECORDS")
    subprocess.run(["wget", "-q", "-O", records_path, records_url])

    edf_list = []
    try:
        with open(records_path, "r") as f:
            edf_list = [l.strip() for l in f.read().splitlines() if l.strip().endswith(".edf")]
    except:
        pass

    source = "RECORDS"
    if len(edf_list) < 5:
        try:
            with open(summary_path, "r") as f:
                content = f.read()
            edf_list = re.findall(r"File Name:\s*(\S+\.edf)", content)
            source = "summary"
        except:
            pass

    n = len(edf_list)
    base_url = f"https://physionet.org/files/chbmit/1.0.0/{pid}/"
    tasks = [(f.strip(), os.path.join(pdir, f.strip()), base_url) for f in edf_list if f.strip()]

    print(f"\n{pid}: {len(tasks)} files ({source}) [6 parallel]")

    def dl_one(args):
        fname, epath, ub = args
        new = False
        if not os.path.exists(epath):
            subprocess.run(["wget","-q","-O",epath, ub+fname],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            new = True
        sz = os.path.getsize(epath) / (1024*1024)
        spath = epath + ".seizures"
        if not os.path.exists(spath):
            subprocess.run(["wget","-q","-O",spath, ub+fname+".seizures"],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return fname, sz, new

    done = 0; newc = 0
    with ThreadPoolExecutor(max_workers=6) as pool:
        futures = [pool.submit(dl_one, t) for t in tasks]
        for f in as_completed(futures):
            fn, sz, is_new = f.result()
            done += 1
            m = " NEW" if is_new else ""
            if is_new: newc += 1
            print(f"  [{done}/{len(tasks)}] {fn} ({sz:.0f} MB){m}")

    elapsed = time.time() - start_time
    pbytes = sum(os.path.getsize(os.path.join(pdir, f)) for f in edf_list)
    total_bytes += pbytes
    print(f"  -> {newc} new, {done-newc} existed | Total: {total_bytes/1e9:.1f} GB | {elapsed/60:.0f} min")

print(f"\n===== DONE ({total_bytes/1e9:.2f} GB in {(time.time()-start_time)/60:.0f} min) =====")

## Cell 3: Parse Seizure Annotations

Reads summary.txt files. No hardcoded timestamps.

In [ ]:
def parse_seizure_summary(summary_path):
    with open(summary_path, "r") as f:
        content = f.read()
    sections = re.split(r"File Name:\s*", content)[1:]
    sm = {}
    for s in sections:
        m = re.match(r"(\S+)", s)
        if not m: continue
        fn = m.group(1)
        st = [int(x) for x in re.findall(r"Seizure\s+Start\s+Time:\s*(\d+)\s*seconds", s)]
        en = [int(x) for x in re.findall(r"Seizure\s+End\s+Time:\s*(\d+)\s*seconds", s)]
        sm[fn] = list(zip(st, en))
    return sm

SEIZURE_MAP = {}
for p in ALL_PATIENTS:
    sp = os.path.join(BASE_DIR, f"{p}-summary.txt")
    if os.path.exists(sp):
        SEIZURE_MAP[p] = parse_seizure_summary(sp)
        n_seiz = sum(1 for v in SEIZURE_MAP[p].values() if v)
        print(f"{p}: {len(SEIZURE_MAP[p])} recordings, {n_seiz} with seizures")
    else:
        SEIZURE_MAP[p] = {}

## Cell 4: bandpass_filter()

Filter full recording ONCE before windowing.

In [ ]:
def bandpass_filter(raw, lowcut=0.5, highcut=40.0, notch_freq=60.0):
    rf = raw.copy()
    rf.filter(lowcut, highcut, fir_design="firwin", verbose=False)
    rf.notch_filter(freqs=notch_freq, verbose=False)
    return rf.get_data()

## Cell 5: create_windows()

30% overlap -> stride=1254 samples (7s @ 256Hz).

In [ ]:
def create_windows(signal, sfreq, window_size_sec=7.0, overlap=0.3):
    ws = int(window_size_sec * sfreq)
    st = int(ws * (1 - overlap))
    total = signal.shape[1]
    wins = []
    for s in range(0, total - ws, st):
        wins.append(signal[:, s:s+ws])
    return np.array(wins)

## Cell 6: create_labels()

Soft overlap: any touch -> seizure=1.

In [ ]:
def create_labels(windows, sfreq, seizure_intervals):
    n = windows.shape[0]
    ws = windows.shape[2]
    stride = int(ws * (1 - 0.3))
    labs = np.zeros(n, dtype=int)
    if not seizure_intervals: return labs
    for i in range(n):
        a = (i*stride)/sfreq
        b = (i*stride+ws)/sfreq
        for st, en in seizure_intervals:
            if b >= st and a <= en:
                labs[i] = 1; break
    return labs

## Cell 7: process_recording()

Orchestrator: load -> filter -> window -> label. Returns raw windows (23 channels).

In [ ]:
def process_recording(edf_path, seizure_intervals,
                      window_size_sec=7.0, overlap=0.3,
                      lowcut=0.5, highcut=40.0, notch_freq=60.0):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    signal = bandpass_filter(raw, lowcut, highcut, notch_freq)
    sfreq = raw.info["sfreq"]
    windows = create_windows(signal, sfreq, window_size_sec, overlap)
    labels = create_labels(windows, sfreq, seizure_intervals)
    return windows, labels

## Cell 8: Process Training Set

Loops over all train patient recordings.

In [ ]:
def collect_recordings(patient_ids, seizure_map, base_dir,
                        window_size_sec=7.0, overlap=0.3):
    all_data, all_labels = [], []
    for pid in patient_ids:
        pdir = os.path.join(base_dir, pid)
        edf_files = sorted([f for f in os.listdir(pdir) if f.endswith(".edf")])
        for edf in edf_files:
            intervals = seizure_map[pid].get(edf, [])
            path = os.path.join(pdir, edf)
            data, labels = process_recording(path, intervals,
                window_size_sec=window_size_sec, overlap=overlap)
            all_data.append(data)
            all_labels.append(labels)
            n = np.sum(labels)
            print(f"  {pid}/{edf} -> {data.shape[0]} windows, {n} seizure")
    return (np.concatenate(all_data, axis=0), np.concatenate(all_labels, axis=0))

print("\n=== PROCESSING TRAIN SET ===")
X_train_1d, y_train_1d = collect_recordings(
    TRAIN_PATIENTS, SEIZURE_MAP, BASE_DIR,
    window_size_sec=WINDOW_SIZE_SEC, overlap=OVERLAP)

print(f"\nTrain: X={X_train_1d.shape}, y={y_train_1d.shape}")
print(f"Seizure: {np.sum(y_train_1d)}, Non-seizure: {len(y_train_1d)-np.sum(y_train_1d)}")

## Cell 9: Process Test Set

Held-out patients only. No data leakage.

In [ ]:
print("\n=== PROCESSING TEST SET ===")
X_test_1d, y_test_1d = collect_recordings(
    TEST_PATIENTS, SEIZURE_MAP, BASE_DIR,
    window_size_sec=WINDOW_SIZE_SEC, overlap=OVERLAP)

print(f"\nTest: X={X_test_1d.shape}, y={y_test_1d.shape}")
print(f"Seizure: {np.sum(y_test_1d)}, Non-seizure: {len(y_test_1d)-np.sum(y_test_1d)}")

## Cell 10: Balance

Keep all seizure windows. Sample 500 non-seizure.

In [ ]:
def bal(data, labels, n_non=500, seed=42):
    np.random.seed(seed)
    si = np.where(labels==1)[0]
    ni = np.where(labels==0)[0]
    cn = np.random.choice(ni, size=min(n_non, len(ni)), replace=False)
    k = np.concatenate([si, cn])
    np.random.shuffle(k)
    return data[k], labels[k]

X_train_1d, y_train_1d = bal(X_train_1d, y_train_1d, NON_SEIZURE_SAMPLES, RANDOM_SEED)
X_test_1d,  y_test_1d  = bal(X_test_1d,  y_test_1d,  NON_SEIZURE_SAMPLES, RANDOM_SEED)
ut, ct = np.unique(y_train_1d, return_counts=True)
print(f"Train balanced: {X_train_1d.shape}, classes={dict(zip(ut, ct))}")
ut, ct = np.unique(y_test_1d, return_counts=True)
print(f"Test balanced:  {X_test_1d.shape}, classes={dict(zip(ut, ct))}")

## Cell 11: Normalize Per-Channel

Each EEG channel independently. Stats from train only.

In [ ]:
X_train_1d = X_train_1d.astype(np.float32)
X_test_1d  = X_test_1d .astype(np.float32)

# Per-channel mean/std from TRAIN only
tm1 = X_train_1d.mean(axis=(0, 2), keepdims=True)
ts1 = X_train_1d.std( axis=(0, 2), keepdims=True)

X_train_1d = (X_train_1d - tm1) / (ts1 + 1e-8)
X_test_1d  = (X_test_1d  - tm1) / (ts1 + 1e-8)
print(f"Normalized per-channel. Train shape: {X_train_1d.shape}")

## Cell 12: Data Augmentation

Conservative: time shift +-200ms, amplitude x0.85-1.15, noise sigma=0.01.
Applied ONLY to seizure windows during training.

In [ ]:
def augment_seizure(windows, seed=None):
    """Augment seizure windows: time shift + scale + noise."""
    if seed is not None: np.random.seed(seed)
    aug = windows.copy()
    B, C, T = aug.shape
    for b in range(B):
        shift = np.random.randint(-51, 51)
        aug[b] = np.roll(aug[b], shift, axis=-1)
        aug[b] *= np.random.uniform(0.85, 1.15)
        aug[b] += np.random.randn(C, T).astype(np.float32) * 0.01
    return aug
print("Augmentation ready.")

## Cell 13: PyTorch Tensors

1D CNN input shape: (batch, 23 channels, 1792 time).

In [ ]:
Xtr1 = torch.tensor(X_train_1d, dtype=torch.float32)
ytr1 = torch.tensor(y_train_1d, dtype=torch.long)
Xte1 = torch.tensor(X_test_1d,  dtype=torch.float32)
yte1 = torch.tensor(y_test_1d,  dtype=torch.long)

tds1 = TensorDataset(Xtr1, ytr1)
eds1 = TensorDataset(Xte1, yte1)

tl1 = DataLoader(tds1, batch_size=BATCH_SIZE, shuffle=True)
el1 = DataLoader(eds1, batch_size=BATCH_SIZE, shuffle=False)

print(f"Input shape: {Xtr1.shape}  (batch, 23, 1792)")
print(f"Train batches: {len(tl1)} | Test: {len(el1)}")

## Cell 14: EEGCNN1D Model

3 Conv1d blocks (k7,k5,k3) + BatchNorm + Dropout + AdaptiveAvgPool1d(16)

```
Block 1: Conv1d(23->32,k=7) -> BN -> ReLU -> MaxPool(4)  (1792->448)
Block 2: Conv1d(32->64,k=5) -> BN -> ReLU -> MaxPool(4)  (448->112)
Block 3: Conv1d(64->128,k=3)-> BN -> ReLU -> AdaptiveAvgPool1d(16)
Flatten -> Dropout(0.4) -> FC(2048->64) -> ReLU -> Dropout(0.3) -> FC(64->2)
```

In [ ]:
class EEGCNN1D(nn.Module):
    def __init__(self, nc=23):
        super().__init__()
        self.conv1 = nn.Conv1d(nc, 32, 7, padding=3)
        self.bn1   = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(4)          # 1792->448

        self.conv2 = nn.Conv1d(32, 64, 5, padding=2)
        self.bn2   = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(4)          # 448->112

        self.conv3 = nn.Conv1d(64, 128, 3, padding=1)
        self.bn3   = nn.BatchNorm1d(128)
        self.adapt = nn.AdaptiveAvgPool1d(16)  # ANY length->16

        self.drop1 = nn.Dropout(0.4)
        self.fc1   = nn.Linear(128 * 16, 64)   # 2048->64
        self.drop2 = nn.Dropout(0.3)
        self.fc2   = nn.Linear(64, 2)           # 64->2

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.adapt(x)
        x = torch.flatten(x, 1)
        x = self.drop1(x)
        x = F.relu(self.fc1(x))
        x = self.drop2(x)
        x = self.fc2(x)
        return x

model_1d = EEGCNN1D()
d = torch.randn(1, 23, 1792)
with torch.no_grad():
    o = model_1d(d)
print(f"Model: {sum(p.numel() for p in model_1d.parameters()):,} params")
print(f"Input {tuple(d.shape)} -> Output {tuple(o.shape)}  OK!")

## Cell 15: Train 1D CNN

Augmentation applied ONLY to seizure windows. 1 conservative copy per window.

In [ ]:
crit1 = nn.CrossEntropyLoss()
opt1 = optim.Adam(model_1d.parameters(), lr=LEARNING_RATE)

for ep in range(NUM_EPOCHS):
    model_1d.train()
    rl = 0.0; nb = 0

    for x, y in tl1:
        sm = (y == 1)
        if sm.any():
            si = x[sm]; sl = y[sm]
            an = augment_seizure(si.numpy())
            at = torch.tensor(an, dtype=torch.float32)
            xa = torch.cat([x, at], 0)
            ya = torch.cat([y, sl], 0)
        else:
            xa = x; ya = y

        opt1.zero_grad()
        o = model_1d(xa)
        l = crit1(o, ya)
        l.backward()
        opt1.step()
        rl += l.item(); nb += 1

    print(f"Epoch {ep+1}/{NUM_EPOCHS}, Loss: {rl/nb:.4f}")

## Cell 16: Evaluate on Unseen Patients

Metrics:
- Accuracy: overall correct %
- Sensitivity (recall): what % of real seizures did we catch?
- Specificity: what % of non-seizures did we correctly reject?

In [ ]:
model_1d.eval()
c, t = 0, 0
ap, at = [], []

with torch.no_grad():
    for x, y in el1:
        o = model_1d(x)
        _, p = torch.max(o, 1)
        ap.extend(p.cpu().numpy())
        at.extend(y.cpu().numpy())
        t += y.size(0)
        c += (p == y).sum().item()

ap = np.array(ap)
at = np.array(at)
print(f"Accuracy: {100*c/t:.2f}% ({c}/{t})")

sm = at == 1
if sm.sum() > 0:
    tp = (ap[sm] == 1).sum()
    fn = (ap[sm] == 0).sum()
    sens = tp / (tp + fn)
    print(f"Sensitivity (recall): {sens:.2f}  (caught {tp}/{tp+fn} seizures)")

nm = at == 0
if nm.sum() > 0:
    tn = (ap[nm] == 0).sum()
    fp = (ap[nm] == 1).sum()
    spec = tn / (tn + fp)
    print(f"Specificity:         {spec:.2f}  (rejected {tn}/{tn+fp} non-seizures)")